# 🏛️ PIT 펀더멘털 팩터 모델 (Point-in-Time, 정직한 밸류·퀄리티)

앞의 `US_factor_model.ipynb`는 펀더멘털을 **현재 스냅샷**으로만 써서 밸류·퀄리티를 백테스트할 수 없었다(look-ahead).
여기서는 **SEC EDGAR companyfacts** 로 각 재무값의 **신고일(filed)** 을 이용해,
"그 날짜에 실제로 알 수 있었던 값"만 사용한다 → **밸류·퀄리티도 look-ahead 없이 정직하게 백테스트**.

## 무엇이 달라졌나
- 밸류(이익수익률·장부수익률)·퀄리티(ROE·이익률)를 **PIT로 재구성** → 5팩터 전부 정직하게 검증
- "남들이 안 하는 데이터 엔지니어링" = 진짜 엣지의 출발점

## 준비물
- **SEC User-Agent** 필요: `.env` 에 `SEC_USER_AGENT=이름 이메일` (없으면 차단될 수 있음)
- 종목마다 SEC 조회 → **수 분 소요**(rate limit 준수)

## 엄격성
- 재무값은 `filed <= 그 시점` 인 것 중 최신 회계기간만 사용 (룩어헤드 0)
- 연간(10-K) 기준 순이익·매출 (분기 TTM은 향후 개선)
- 월간 리밸런스, 거래비용 반영, 상위 20% 롱

> ⚠️ 투자 자문 아님. 교육/연구용.

In [ ]:
# =====================================================================
# ⚙️ Cell 1: 부트스트랩(패키지 import) + 설정
# =====================================================================
import sys, os
_here = os.getcwd()
for _c in [_here, os.path.abspath(os.path.join(_here, '..', '..')), os.path.abspath(os.path.join(_here, '..'))]:
    if os.path.isdir(os.path.join(_c, 'trend_system')):
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False

from trend_system.fundamentals import load_fundamentals, pit_factor_row

UNIVERSE = [
    'AAPL','MSFT','GOOGL','AMZN','META','NVDA','AVGO','ORCL','CRM','ADBE','AMD','QCOM','TXN','INTC','CSCO',
    'JPM','BAC','WFC','GS','MS','C','V','MA','AXP','BLK',
    'UNH','JNJ','LLY','PFE','ABBV','MRK','TMO','ABT',
    'PG','KO','PEP','WMT','COST','HD','MCD','NKE','SBUX','DIS',
    'XOM','CVX','CAT','BA','HON','GE','UPS','NFLX','T','VZ',
]
START='2012-01-01'; END=None; COST=0.0005; ANN=252; REBAL=21; TOP_Q=0.20
print(f'유니버스 {len(UNIVERSE)} | 리밸런스 {REBAL}일 | 상위 {TOP_Q:.0%} | 비용 {COST*100:.2f}%')

In [ ]:
# =====================================================================
# 📥 Cell 2: 가격 패널
# =====================================================================
def fetch_close(tickers):
    frames={}
    for t in tickers:
        try:
            df=yf.download(t,start=START,end=END,auto_adjust=True,progress=False)
            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(0)
            s=df['Close'].dropna()
            if len(s)>300: frames[t]=s
        except Exception: pass
    return pd.DataFrame(frames)

print('가격 수집 중...')
close=fetch_close(UNIVERSE)
close=close.dropna(axis=1, thresh=int(len(close)*0.8)).dropna()
print(f'가격 패널: {close.shape[1]}종목 × {close.shape[0]}일')

In [ ]:
# =====================================================================
# 🧮 Cell 3: 가격기반 팩터 (PIT)
# =====================================================================
rets=close.pct_change()
f_mom=close.shift(21)/close.shift(252)-1
f_lowvol=-(rets.rolling(126).std())
f_rev=-(close.pct_change(21))
PRICE_FACTORS={'Momentum':f_mom,'LowVol':f_lowvol,'Reversal':f_rev}
print('가격 팩터:', list(PRICE_FACTORS.keys()))

In [ ]:
# =====================================================================
# 🏛️ Cell 4: SEC EDGAR PIT 펀더멘털 로드 (느림! 종목별 조회, 수 분)
#   .env 의 SEC_USER_AGENT 필요
# =====================================================================
fund = load_fundamentals(list(close.columns))
print('로드된 종목 수:', len(fund))

In [ ]:
# =====================================================================
# 🧪 Cell 5: 5팩터(가격3+PIT밸류/퀄리티) 크로스섹셔널 백테스트
# =====================================================================
def zscore_row(row):
    m,s=row.mean(),row.std()
    return (row-m)/s if (s and s>0) else row*0.0

def factor_scores_at(d):
    zs={name:zscore_row(f.loc[d]) for name,f in PRICE_FACTORS.items()}
    ey,by,roe,margin=pit_factor_row(fund,list(close.columns),d,close.loc[d])
    val =pd.concat([ey,by],axis=1).mean(axis=1)  if (len(ey)+len(by))  else pd.Series(dtype=float)
    qual=pd.concat([roe,margin],axis=1).mean(axis=1) if (len(roe)+len(margin)) else pd.Series(dtype=float)
    zs['Value']  =zscore_row(val.reindex(close.columns))
    zs['Quality']=zscore_row(qual.reindex(close.columns))
    return zs

dates=close.index
rebal_idx=list(range(252,len(dates)-1,REBAL))
print(f'리밸런스 {len(rebal_idx)}회 · PIT 팩터 점수 계산 중...')
SCORES={dates[i]:factor_scores_at(dates[i]) for i in rebal_idx}

def perf(pr):
    r=np.asarray(pr,float); out={'CAGR':np.nan,'Sharpe':0.0,'MDD':0.0,'Calmar':np.nan}
    if len(r)==0: return out
    ppy=ANN/REBAL; eq=np.cumprod(1+r); peak=np.maximum.accumulate(eq)
    out['CAGR']=float(eq[-1]**(ppy/len(r))-1) if eq[-1]>0 else np.nan
    out['Sharpe']=float(r.mean()/r.std()*np.sqrt(ppy)) if r.std()>0 else 0.0
    out['MDD']=float(((eq-peak)/peak).min())
    out['Calmar']=float(out['CAGR']/abs(out['MDD'])) if out['MDD']<0 else np.nan
    return out

def backtest(get_score):
    port,bench,dd=[],[],[]; prev=set(); turn=0.0
    for i in rebal_idx:
        d=dates[i]; score=get_score(d).dropna()
        if len(score)<10: continue
        k=max(int(len(score)*TOP_Q),1); picks=list(score.sort_values(ascending=False).head(k).index)
        j=min(i+REBAL,len(dates)-1); d2=dates[j]
        hr=float((close.loc[d2,picks]/close.loc[d,picks]-1).mean())
        new=set(picks); ntr=len(new-prev)+len(prev-new); turn+=ntr
        port.append(hr-(ntr/max(len(new),1))*COST)
        bench.append(float((close.loc[d2]/close.loc[d]-1).mean())); dd.append(d2); prev=new
    return pd.Series(port,index=dd), pd.Series(bench,index=dd), turn

def composite(d):
    zs=SCORES[d]; return sum(z.reindex(close.columns).fillna(0) for z in zs.values())/len(zs)

port_ret,bench_ret,turn=backtest(composite)
mp,mb=perf(port_ret.values),perf(bench_ret.values)
print('\n=== 5팩터 복합 백테스트 (PIT, 비용반영) ===')
print(f'{"":18}{"CAGR":>9}{"Sharpe":>9}{"MDD":>9}{"Calmar":>9}')
print(f'{"복합 상위20% 롱":18}{mp["CAGR"]*100:8.1f}%{mp["Sharpe"]:9.2f}{mp["MDD"]*100:8.1f}%{mp["Calmar"]:9.2f}')
print(f'{"유니버스 균등보유":18}{mb["CAGR"]*100:8.1f}%{mb["Sharpe"]:9.2f}{mb["MDD"]*100:8.1f}%{mb["Calmar"]:9.2f}')

In [ ]:
# =====================================================================
# 🔬 Cell 6: 팩터별 기여도 (밸류·퀄리티 포함, 이제 전부 PIT 백테스트)
# =====================================================================
def getter(name): return lambda d: SCORES[d][name].reindex(close.columns)
print('=== 팩터별 단독 성과 (상위20% 롱, PIT) ===')
print(f'{"팩터":12}{"CAGR":>9}{"Sharpe":>9}{"MDD":>9}')
curves={}
for name in ['Momentum','LowVol','Reversal','Value','Quality']:
    pr,_,_=backtest(getter(name)); curves[name]=pr; mm=perf(pr.values)
    print(f'{name:12}{mm["CAGR"]*100:8.1f}%{mm["Sharpe"]:9.2f}{mm["MDD"]*100:8.1f}%')
mc=perf(port_ret.values)
print(f'{"복합(5팩터)":12}{mc["CAGR"]*100:8.1f}%{mc["Sharpe"]:9.2f}{mc["MDD"]*100:8.1f}%')

plt.figure(figsize=(13,6))
(1+port_ret).cumprod().plot(label='복합 5팩터', lw=2.4, color='darkred')
(1+bench_ret).cumprod().plot(label='유니버스 균등보유', lw=1.4, ls='--', color='gray')
for name,pr in curves.items():
    (1+pr).cumprod().plot(label=name, lw=1, alpha=0.7)
plt.yscale('log'); plt.title('PIT 5팩터 크로스섹셔널 — 자산곡선(로그)')
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# =====================================================================
# 🔮 Cell 7: 오늘의 랭킹 (5팩터 전부 PIT 반영 — 이제 forward도 백테스트와 동일 로직)
# =====================================================================
today=close.index[-1]
zs=factor_scores_at(today)
comb=sum(z.reindex(close.columns).fillna(0) for z in zs.values())/len(zs)
rank=pd.DataFrame({n: zs[n].reindex(close.columns).round(2) for n in zs})
rank['combined']=comb.round(3)
rank=rank.sort_values('combined',ascending=False)
k=max(int(len(rank)*TOP_Q),1)
print('='*74)
print(f' 🔮 오늘의 5팩터 랭킹 (기준일 {today.date()}) — 상위 {k}종목 롱 후보')
print('='*74)
print(rank.head(k).to_string())
print('='*74)
print(' 🏆 롱 후보:', ', '.join(rank.head(k).index.tolist()))

## 🧾 결과 해석 & 다음

### 백테스트에서 볼 것 (이제 5팩터 전부 정직)
- **복합 Sharpe > 균등보유** 이면 팩터 순위매기기가 실제로 기여.
- **밸류·퀄리티 단독 Sharpe** — 대형주에선 약할 수 있으나, 가격팩터와 **다른 시점에** 작동하면 복합에 기여.
- 복합이 개별 최고보다 높으면 → 진짜 팩터 분산.

### 여전한 한계
1. **순이익/매출은 연간(10-K) 기준** — 분기 TTM보다 반응이 느림(값이 1년에 한 번 갱신). 향후 4분기 합산 TTM으로 개선 가능.
2. **생존 편향** — 현재 유니버스라 과거 백테스트 낙관적. point-in-time 구성종목 이력이 정석.
3. **대형주는 이미 arbitraged** — 소형주/해외에서 팩터가 더 강함.

### 다음 장
1. **분기 TTM 순이익** (companyfacts 분기값 4개 합산) → 밸류·퀄리티 반응성↑
2. **유니버스 확대**(S&P500) + **생존편향 보정**
3. **롱숏 중립화**: 상위 롱 − 하위 숏 → 시장 베타 제거, 순수 팩터 알파 측정
4. **어닝 서프라이즈/리비전**(PEAD) 팩터 추가 — "느리게 반영되는 정보"

> ⚠️ 파라미터는 관습값 고정. 최적화 시작하면 과적합. 견고성이 곧 가치.